# MedFrameQA 最终验证

这份 notebook 专门服务于论文补实验：

- 对 `selected_generation` 与 `gen_0` 跑 post-hoc final test
- 导出逐题预测
- 生成 `modality / image_count` breakdown
- 计算 paired bootstrap


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path("/gluon4/xl693/evolve")
RESULTS_ROOT = ROOT / "results"
OUTPUT_DIR = ROOT / "paper_analysis_output"
METHODS = ["fixed", "reasoning", "order_vote", "order_rerank", "order_vote_plus"]
POSTHOC_SCRIPT = ROOT / "medframeqa_posthoc_eval.py"
BOOTSTRAP_SCRIPT = ROOT / "medframeqa_paired_bootstrap.py"

print("ROOT:", ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("METHODS:", METHODS)


In [ ]:
from collections import Counter

from medframeqa_runtime import get_image_columns, load_medframeqa_dataset

dataset = load_medframeqa_dataset(include_images=True)
image_count_counter = Counter(len(get_image_columns(sample)) for sample in dataset)
print("dataset image_count_counter:", dict(sorted(image_count_counter.items())))
assert set(image_count_counter).issubset({2, 3, 4, 5}), image_count_counter


In [ ]:
# 这格会真正运行 post-hoc final test，成本较高。
cmd = [
    sys.executable,
    str(POSTHOC_SCRIPT),
    "--results-root",
    str(RESULTS_ROOT),
    "--protocol-mode",
    "independent_final_test",
    "--methods",
    *METHODS,
    "--targets",
    "selected",
    "gen0",
]
print("Running posthoc command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
posthoc_rows = json.loads((OUTPUT_DIR / "posthoc_eval" / "posthoc_eval_rows.json").read_text())
selected_vs_gen0 = json.loads((OUTPUT_DIR / "posthoc_eval" / "selected_vs_gen0.json").read_text())

print("selected_vs_gen0:")
for row in selected_vs_gen0:
    print(row)


In [ ]:
protocol_checks = {}
for row in posthoc_rows:
    if row["target"] != "selected":
        continue
    check_path = Path(row["output_dir"]) / "protocol_checks.json"
    protocol_checks[row["method"]] = json.loads(check_path.read_text())

print("protocol_checks:")
for method, payload in protocol_checks.items():
    print(method, payload)


In [ ]:
import matplotlib.pyplot as plt

delta_rows = [row for row in selected_vs_gen0 if row.get("delta_selected_minus_gen0") is not None]
methods = [row["method"] for row in delta_rows]
deltas = [row["delta_selected_minus_gen0"] for row in delta_rows]

plt.figure(figsize=(9, 4))
plt.bar(methods, deltas)
plt.axhline(0.0, color="black", linewidth=1)
plt.ylabel("selected - gen0 final test score")
plt.title("Evolution gain on independent final test")
plt.tight_layout()
plt.show()


In [ ]:
selected_scores = {row["method"]: row["selected_final_test_score"] for row in selected_vs_gen0}
pairs = ["order_vote,fixed", "order_vote,order_rerank"]
if (
    selected_scores.get("order_vote_plus") is not None
    and selected_scores.get("order_rerank") is not None
    and selected_scores.get("order_vote") is not None
    and selected_scores["order_vote_plus"] >= selected_scores["order_rerank"]
    and (selected_scores["order_vote"] - selected_scores["order_vote_plus"]) <= 0.01
):
    pairs.append("order_vote_plus,order_vote")

cmd = [
    sys.executable,
    str(BOOTSTRAP_SCRIPT),
    "--results-root",
    str(RESULTS_ROOT),
    "--eval-name",
    "posthoc_selected_final_test",
    "--pairs",
    *pairs,
]
print("Running bootstrap command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
bootstrap_rows = json.loads((OUTPUT_DIR / "bootstrap_posthoc_selected_final_test.json").read_text())
print("bootstrap_rows:")
for row in bootstrap_rows:
    print(row)


In [ ]:
from collections import defaultdict

breakdown_dir_map = {}
for row in posthoc_rows:
    if row["target"] == "selected":
        breakdown_dir_map[row["method"]] = Path(row["output_dir"])

modality_tables = {}
image_count_tables = {}
for method in ["fixed", "order_vote", "order_rerank", "order_vote_plus"]:
    out_dir = breakdown_dir_map.get(method)
    if out_dir is None:
        continue
    modality_tables[method] = json.loads((out_dir / "breakdown_modality.json").read_text())
    image_count_tables[method] = json.loads((out_dir / "breakdown_image_count.json").read_text())

print("modality_tables:")
for method, rows in modality_tables.items():
    print(method, rows)

print("\nimage_count_tables:")
for method, rows in image_count_tables.items():
    print(method, rows)
